# Source Code GTWR IRIS L3mao 

## 0. Import Library

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
import matplotlib.pyplot as plt

## 1. GTWR

In [9]:
df = pd.read_csv("C:/Uner/Lomba/HSE/Essai/data gtwr.csv")

def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

df['hujan_norm'] = normalize(df['hujan_mm'])
df['kelembaban_norm'] = normalize(df['kelembaban_tanah'])
df['miskin_norm'] = normalize(df['persentase_penduduk_miskin'])

df['Skor_Risiko'] = (df['hujan_norm'] * 0.5 + 
                     df['kelembaban_norm'] * 0.3 + 
                     df['miskin_norm'] * 0.2) * 100

df['tanggal'] = pd.to_datetime(df['tanggal'])
df['t'] = (df['tanggal'] - df['tanggal'].min()).dt.days

y = df['Skor_Risiko'].values.reshape((-1, 1))
X = df[['hujan_mm', 'suhu_rata', 'kelembaban_tanah', 'persentase_penduduk_miskin']].values
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

tau = 1.0 
coords_s = df[['lat', 'lon']].values
coords_t = df['t'].values.reshape(-1, 1)
scaler_coords = StandardScaler()
coords_s_scaled = scaler_coords.fit_transform(coords_s)
coords_t_scaled = scaler_coords.fit_transform(coords_t)
coords_st = np.hstack([coords_s_scaled, coords_t_scaled]) # Tau otomatis 1:1 via Z-score

selector = Sel_BW(coords_st, y, X_scaled, kernel='bisquare', fixed=False)
bw_optimal = selector.search(criterion='AICc')
print(f"Bandwidth Optimal: {bw_optimal}")

model = GWR(coords_st, y, X_scaled, bw=bw_optimal, kernel='bisquare', fixed=False)
results = model.fit()

print(f"R-Square: {results.R2}")
print(f"Adjusted R-Square: {results.adj_R2}")

results.summary()



Bandwidth Optimal: 1462.0
R-Square: 1.0
Adjusted R-Square: 1.0
Model type                                                         Gaussian
Number of observations:                                               13870
Number of covariates:                                                     5

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                              0.000
Log-likelihood:                                                  422095.022
AIC:                                                            -844180.045
AICc:                                                           -844178.039
BIC:                                                            -132237.209
R2:                                                                   1.000
Adj. R2:                                                              1.000

Variable                              Est.         SE  t(Est/SE)    p-val

## 2. Detailed GTWR

In [10]:
print("\n=== RINGKASAN MODEL GTWR ===")
try:
    results.summary()
except Exception as e:
    print(f"Summary tidak dapat ditampilkan: {e}")

try:
    condition_number = results.local_collinearity()
    print("\n=== DIAGNOSA MULTIKOLINEARITAS LOKAL ===")
    print(f"Rata-rata Condition Number: {np.mean(condition_number[0]):.2f}")
    print(f"Maksimum Condition Number:  {np.max(condition_number[0]):.2f}")
    if np.max(condition_number[0]) > 30:
        print("PERINGATAN: Terdeteksi potensi multikolinearitas lokal (CN > 30).")
        print("Solusi: Coba kurangi variabel yang berkorelasi tinggi atau gunakan metode seleksi variabel.")
    else:
        print("STATUS: Aman. Tidak terdeteksi multikolinearitas lokal yang serius (CN < 30).")
except Exception as e:
    print(f"Gagal menghitung collinearity: {e}")


coef_names = ['Intercept', 'B_Hujan', 'B_Suhu', 'B_Lembab', 'B_Miskin']
df_results = pd.concat([df, pd.DataFrame(results.params, columns=coef_names)], axis=1)
df_results.to_csv('Hasil_GTWR_Final.csv', index=False)
print("File 'Hasil_GTWR_Final.csv' telah siap untuk divisualisasikan.")


=== RINGKASAN MODEL GTWR ===
Model type                                                         Gaussian
Number of observations:                                               13870
Number of covariates:                                                     5

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                              0.000
Log-likelihood:                                                  422095.022
AIC:                                                            -844180.045
AICc:                                                           -844178.039
BIC:                                                            -132237.209
R2:                                                                   1.000
Adj. R2:                                                              1.000

Variable                              Est.         SE  t(Est/SE)    p-value
------------------------------

## 3. Detailed GTWR

In [11]:
sns.set_theme(style="whitegrid")

df = pd.read_csv("C:/Uner/Lomba/HSE/Essai/Hasil_GTWR_Final.csv")
df['tanggal'] = pd.to_datetime(df['tanggal'])

def plot_tren_risiko(df, nama_provinsi):
    prov_df = df[df['nama_provinsi'] == nama_provinsi].sort_values('tanggal')
    plt.plot(prov_df['tanggal'], prov_df['Skor_Risiko'], color='#e74c3c', linewidth=2)
    plt.fill_between(prov_df['tanggal'], prov_df['Skor_Risiko'], alpha=0.2, color='#e74c3c')
    plt.title(f'Tren Skor Risiko Spasiotemporal: {nama_provinsi} (2024)', fontsize=14, pad=15)
    plt.xlabel('Bulan', fontsize=12)
    plt.ylabel('Skor Risiko (0-100)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('tren_risiko_provinsi.png')
    plt.close()

def plot_distribusi_beta(df):
    beta_cols = ['B_Hujan', 'B_Suhu', 'B_Lembab', 'B_Miskin']
    labels = ['Curah Hujan', 'Suhu', 'Kelembaban', 'Kemiskinan']
    
    plt.boxplot([df[col] for col in beta_cols], labels=labels)
    plt.title('Distribusi Koefisien Lokal (Beta (β)) Hasil Pemodelan GTWR', fontsize=14, pad=15)
    plt.ylabel('Nilai Koefisien', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('distribusi_koefisien.png')
    plt.close()

def plot_ranking_provinsi(df):
    avg_risk = df.groupby('nama_provinsi')['Skor_Risiko'].mean().sort_values(ascending=False)
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(avg_risk)))
    
    plt.bar(avg_risk.index, avg_risk.values, color=colors)
    plt.title('Peringkat Rata-rata Skor Risiko per Provinsi (Tahun 2024)', fontsize=14, pad=15)
    plt.ylabel('Rata-rata Skor Risiko', fontsize=12)
    plt.xticks(rotation=90, fontsize=9)
    plt.tight_layout()
    plt.savefig('ranking_risiko_provinsi.png')
    plt.close()

plot_tren_risiko(df, 'SUMATERA UTARA') # Contoh untuk Sumatera Utara
plot_distribusi_beta(df)
plot_ranking_provinsi(df)

print("Visualisasi berhasil dibuat: 'tren_risiko_provinsi.png', 'distribusi_koefisien.png', dan 'ranking_risiko_provinsi.png'")

C:\Users\Faiz Iqbal\AppData\Local\Temp\ipykernel_24360\388996969.py:22: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([df[col] for col in beta_cols], labels=labels)


Visualisasi berhasil dibuat: 'tren_risiko_provinsi.png', 'distribusi_koefisien.png', dan 'ranking_risiko_provinsi.png'
